<a href="https://colab.research.google.com/github/rasia92015-byte/Boltz2_colab/blob/main/notebooks/databases/curated/003_1_fetch_active_domain_from_uniprot_id.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

for kinase

In [ ]:
need dedup

In [ ]:
import time
import requests
from pathlib import Path
from openpyxl import load_workbook

INPUT_XLSX = Path("Untitled spreadsheet.xlsx")
OUTPUT_XLSX = Path("Untitled spreadsheet_with_kinase_domains.xlsx")

# To exactly match your original function, use:
# MATCH_TERMS = ("kinase",)
# This broader set also captures lipid-kinase domains labeled PI3K/PI4K by UniProt.
MATCH_TERMS = ("kinase", "pi3k", "pi4k")

def find_uniprot_id_column(ws):
    headers = [cell.value for cell in ws[1]]
    for idx, value in enumerate(headers, start=1):
        if value and str(value).strip().lower() in {"uniprot_id", "uniprot id", "accession", "uniprot"}:
            return idx
    # Fallback: first column
    return 1

def get_kinase_domains_from_uniprot(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    seq = data["sequence"]["value"]
    domains = []

    for feat in data.get("features", []):
        if feat.get("type") != "Domain":
            continue

        desc = feat.get("description", "")
        desc_l = desc.lower()

        if any(term in desc_l for term in MATCH_TERMS):
            start = int(feat["location"]["start"]["value"])
            end = int(feat["location"]["end"]["value"])

            domain_start = max(1, start - 10)
            domain_end = min(len(seq), end + 10)

            domains.append({
                "description": desc,
                "start": start,
                "end": end,
                "sequence": seq[start - 1:end],
                "domain_start": domain_start,
                "domain_end": domain_end,
                "domain": seq[domain_start - 1:domain_end],
                "source_url": url,
                "status": "OK",
            })

    if not domains:
        domains.append({
            "description": "",
            "start": "",
            "end": "",
            "sequence": "",
            "domain_start": "",
            "domain_end": "",
            "domain": "",
            "source_url": url,
            "status": "No matching kinase Domain feature found",
        })

    return domains

def main():
    wb = load_workbook(INPUT_XLSX)
    input_ws = wb["Sheet1"] if "Sheet1" in wb.sheetnames else wb.active
    id_col = find_uniprot_id_column(input_ws)

    accessions = []
    for row in range(2, input_ws.max_row + 1):
        value = input_ws.cell(row=row, column=id_col).value
        if value:
            accessions.append(str(value).strip())

    if "Kinase_domains" in wb.sheetnames:
        del wb["Kinase_domains"]

    out_ws = wb.create_sheet("Kinase_domains")
    out_ws.append([
        "UniProt_ID",
        "domain_description",
        "start",
        "end",
        "sequence",
        "domain_start",
        "domain_end",
        "domain_start_minus_10_to_end_plus_10",
        "source_url",
        "status",
    ])

    for accession in accessions:
        print(f"Fetching {accession} ...")
        try:
            domains = get_kinase_domains_from_uniprot(accession)
            for d in domains:
                out_ws.append([
                    accession,
                    d["description"],
                    d["start"],
                    d["end"],
                    d["sequence"],
                    d["domain_start"],
                    d["domain_end"],
                    d["domain"],
                    d["source_url"],
                    d["status"],
                ])
        except Exception as e:
            out_ws.append([
                accession, "", "", "", "", "", "",
                "",
                f"https://rest.uniprot.org/uniprotkb/{accession}.json",
                f"ERROR: {type(e).__name__}: {e}",
            ])

        # Be polite to the API.
        time.sleep(0.2)

    # Basic formatting
    for col in range(1, out_ws.max_column + 1):
        out_ws.cell(row=1, column=col).style = "Headline 3"

    widths = {
        "A": 14, "B": 24, "C": 10, "D": 10, "E": 60,
        "F": 12, "G": 12, "H": 70, "I": 50, "J": 42,
    }
    for col, width in widths.items():
        out_ws.column_dimensions[col].width = width

    out_ws.freeze_panes = "A2"
    wb.save(OUTPUT_XLSX)
    print(f"Done: {OUTPUT_XLSX}")

if __name__ == "__main__":
    main()



Fetching P10827 ...
Fetching P10827 ...
Fetching P10828 ...
Fetching P10828 ...
Fetching P10276 ...
Fetching P10276 ...
Fetching P10826 ...
Fetching P10826 ...
Fetching P13631 ...
Fetching P13631 ...
Fetching Q07869 ...
Fetching Q07869 ...
Fetching Q03181 ...
Fetching Q03181 ...
Fetching P37231 ...
Fetching P37231 ...
Fetching P20393 ...
Fetching P20393 ...
Fetching Q14995 ...
Fetching P35398 ...
Fetching P35398 ...
Fetching P51449 ...
Fetching P55055 ...
Fetching P55055 ...
Fetching Q13133 ...
Fetching Q13133 ...
Fetching Q96RI1 ...
Fetching Q96RI1 ...
Fetching P11473 ...
Fetching P11473 ...
Fetching O75469 ...
Fetching O75469 ...
Fetching Q14994 ...
Fetching Q14994 ...
Fetching P19793 ...
Fetching P19793 ...
Fetching P03372 ...
Fetching P03372 ...
Fetching Q92731 ...
Fetching Q92731 ...
Fetching P04150 ...
Fetching P04150 ...
Fetching P08235 ...
Fetching P08235 ...
Fetching P06401 ...
Fetching P06401 ...
Fetching P10275 ...
Fetching P10275 ...
Done: 2B_Nuclear_Ligand_Master_SMILES_do

for Nuclear

In [ ]:
import time
import requests
from pathlib import Path
from openpyxl import load_workbook

INPUT_XLSX = Path("test.xlsx")
OUTPUT_XLSX = Path("2B_Nuclear_Ligand_Master_SMILES_domains.xlsx")

# To exactly match your original function, use:
# MATCH_TERMS = ("kinase",)
# This broader set also captures lipid-kinase domains labeled PI3K/PI4K by UniProt.
MATCH_TERMS = ("kinase", "pi3k", "pi4k")

def find_uniprot_id_column(ws):
    headers = [cell.value for cell in ws[1]]
    for idx, value in enumerate(headers, start=1):
        if value and str(value).strip().lower() in {"uniprot_id", "uniprot id", "accession", "uniprot"}:
            return idx
    # Fallback: first column
    return 1

def get_kinase_domains_from_uniprot(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    seq = data["sequence"]["value"]
    print(seq)
    domains = []
    domains.append({


                "domain_start": 1,
                "domain_end": len(seq),
                "domain": seq,

                "status": "OK",
            })




    if not domains:
        domains.append({
            "description": "",
            "start": "",
            "end": "",
            "sequence": "",
            "domain_start": "",
            "domain_end": "",
            "domain": "",
            "source_url": url,
            "status": "No matching kinase Domain feature found",
        })

    return domains

def main():
    wb = load_workbook(INPUT_XLSX)
    input_ws = wb["Sheet1"] if "Sheet1" in wb.sheetnames else wb.active
    id_col = find_uniprot_id_column(input_ws)

    accessions = []
    for row in range(2, input_ws.max_row + 1):
        value = input_ws.cell(row=row, column=id_col).value
        if value:
            accessions.append(str(value).strip())

    if "Kinase_domains" in wb.sheetnames:
        del wb["Kinase_domains"]

    out_ws = wb.create_sheet("Kinase_domains")
    out_ws.append([
        "UniProt_ID",
        "domain_description",
        "start",
        "end",
        "sequence",
        "domain_start",
        "domain_end",
        "domain_start_minus_10_to_end_plus_10",
        "source_url",
        "status",
    ])

    for accession in accessions:
        print(f"Fetching {accession} ...")
        try:
            domains = get_kinase_domains_from_uniprot(accession)
            for d in domains:
                out_ws.append([
                    accession,
                    d["description"],
                    d["start"],
                    d["end"],
                    d["sequence"],
                    d["domain_start"],
                    d["domain_end"],
                    d["domain"],
                    d["source_url"],
                    d["status"],
                ])
        except Exception as e:
            out_ws.append([
                accession, "", "", "", "", "", "",
                "",
                f"https://rest.uniprot.org/uniprotkb/{accession}.json",
                f"ERROR: {type(e).__name__}: {e}",
            ])

        # Be polite to the API.
        time.sleep(0.2)

    # Basic formatting
    for col in range(1, out_ws.max_column + 1):
        out_ws.cell(row=1, column=col).style = "Headline 3"

    widths = {
        "A": 14, "B": 24, "C": 10, "D": 10, "E": 60,
        "F": 12, "G": 12, "H": 70, "I": 50, "J": 42,
    }
    for col, width in widths.items():
        out_ws.column_dimensions[col].width = width

    out_ws.freeze_panes = "A2"
    wb.save(OUTPUT_XLSX)
    print(f"Done: {OUTPUT_XLSX}")

if __name__ == "__main__":
    main()



Fetching P28074 ...
MALASVLERPLPVNQRGFFGLGGRADLLDLGPGSLSDGLSLAAPGWGVPEEPGIEMLHGTTTLAFKFRHGVIVAADSRATAGAYIASQTVKKVIEINPYLLGTMAGGAADCSFWERLLARQCRIYELRNKERISVAAASKLLANMVYQYKGMGLSMGTMICGWDKRGPGLYYVDSEGNRISGATFSVGSGSVYAYGVMDRGYSYDLEVEQAYDLARRAIYQATYRDAYSGGAVNLYHVREDGWIRVSSDNVADLHEKYSGSTP
Fetching P28062 ...
MALLDVCGAPRGQRPESALPVAGSGRRSDPGHYSFSMRSPELALPRGMQPTEFFQSLGGDGERNVQIEMAHGTTTLAFKFQHGVIAAVDSRASAGSYISALRVNKVIEINPYLLGTMSGCAADCQYWERLLAKECRLYYLRNGERISVSAASKLLSNMMCQYRGMGLSMGSMICGWDKKGPGLYYVDEHGTRLSGNMFSTGSGNTYAYGVMDSGYRPNLSPEEAYDLGRRAIAYATHRDSYSGGVVNMYHMKEDGWVKVESTDVSDLLHQYREANQ
Done: 2B_Nuclear_Ligand_Master_SMILES_domains.xlsx


need dedup

In [ ]:
import time
import requests
from pathlib import Path
from openpyxl import load_workbook

INPUT_XLSX = Path("2B_Nuclear_Ligand_Master_SMILES.xlsx")
OUTPUT_XLSX = Path("2B_Nuclear_Ligand_Master_SMILES_domains.xlsx")

# To exactly match your original function, use:
# MATCH_TERMS = ("kinase",)
# This broader set also captures lipid-kinase domains labeled PI3K/PI4K by UniProt.
# This broader set also captures Nuclear R by UniProt.
MATCH_TERMS = ("NR LBD")

def find_uniprot_id_column(ws):
    headers = [cell.value for cell in ws[1]]
    for idx, value in enumerate(headers, start=1):
        if value and str(value).strip().lower() in {"uniprot_id", "uniprot id", "accession", "uniprot"}:
            return idx
    # Fallback: first column
    return 1

def get_kinase_domains_from_uniprot(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    seq = data["sequence"]["value"]
    domains = []

    for feat in data.get("features", []):
        if feat.get("type") != "Domain":
            continue

        desc = feat.get("description", "")
        desc_l = desc.lower()

        if any(term in desc_l for term in MATCH_TERMS):
            start = int(feat["location"]["start"]["value"])
            end = int(feat["location"]["end"]["value"])

            domain_start = max(1, start - 10)
            domain_end = min(len(seq), end + 10)

            domains.append({
                "description": desc,
                "start": start,
                "end": end,
                "sequence": seq[start - 1:end],
                "domain_start": domain_start,
                "domain_end": domain_end,
                "domain": seq[domain_start - 1:domain_end],
                "source_url": url,
                "status": "OK",
            })

    if not domains:
        domains.append({
            "description": "",
            "start": "",
            "end": "",
            "sequence": "",
            "domain_start": "",
            "domain_end": "",
            "domain": "",
            "source_url": url,
            "status": "No matching kinase Domain feature found",
        })

    return domains

def main():
    wb = load_workbook(INPUT_XLSX)
    input_ws = wb["Sheet1"] if "Sheet1" in wb.sheetnames else wb.active
    id_col = find_uniprot_id_column(input_ws)

    accessions = []
    for row in range(2, input_ws.max_row + 1):
        value = input_ws.cell(row=row, column=id_col).value
        if value:
            accessions.append(str(value).strip())

    if "Kinase_domains" in wb.sheetnames:
        del wb["Kinase_domains"]

    out_ws = wb.create_sheet("Kinase_domains")
    out_ws.append([
        "UniProt_ID",
        "domain_description",
        "start",
        "end",
        "sequence",
        "domain_start",
        "domain_end",
        "domain_start_minus_10_to_end_plus_10",
        "source_url",
        "status",
    ])

    for accession in accessions:
        print(f"Fetching {accession} ...")
        try:
            domains = get_kinase_domains_from_uniprot(accession)
            for d in domains:
                out_ws.append([
                    accession,
                    d["description"],
                    d["start"],
                    d["end"],
                    d["sequence"],
                    d["domain_start"],
                    d["domain_end"],
                    d["domain"],
                    d["source_url"],
                    d["status"],
                ])
        except Exception as e:
            out_ws.append([
                accession, "", "", "", "", "", "",
                "",
                f"https://rest.uniprot.org/uniprotkb/{accession}.json",
                f"ERROR: {type(e).__name__}: {e}",
            ])

        # Be polite to the API.
        time.sleep(0.2)

    # Basic formatting
    for col in range(1, out_ws.max_column + 1):
        out_ws.cell(row=1, column=col).style = "Headline 3"

    widths = {
        "A": 14, "B": 24, "C": 10, "D": 10, "E": 60,
        "F": 12, "G": 12, "H": 70, "I": 50, "J": 42,
    }
    for col, width in widths.items():
        out_ws.column_dimensions[col].width = width

    out_ws.freeze_panes = "A2"
    wb.save(OUTPUT_XLSX)
    print(f"Done: {OUTPUT_XLSX}")

if __name__ == "__main__":
    main()



Fetching P10827 ...
Fetching P10827 ...
Fetching P10828 ...
Fetching P10828 ...
Fetching P10276 ...
Fetching P10276 ...
Fetching P10826 ...
Fetching P10826 ...
Fetching P13631 ...
Fetching P13631 ...
Fetching Q07869 ...
Fetching Q07869 ...
Fetching Q03181 ...
Fetching Q03181 ...
Fetching P37231 ...
Fetching P37231 ...
Fetching P20393 ...
Fetching P20393 ...
Fetching Q14995 ...
Fetching P35398 ...
Fetching P35398 ...
Fetching P51449 ...
Fetching P55055 ...
Fetching P55055 ...
Fetching Q13133 ...
Fetching Q13133 ...
Fetching Q96RI1 ...
Fetching Q96RI1 ...
Fetching P11473 ...
Fetching P11473 ...
Fetching O75469 ...
Fetching O75469 ...
Fetching Q14994 ...
Fetching Q14994 ...
Fetching P19793 ...
Fetching P19793 ...
Fetching P03372 ...
Fetching P03372 ...
Fetching Q92731 ...
Fetching Q92731 ...
Fetching P04150 ...
Fetching P04150 ...
Fetching P08235 ...
Fetching P08235 ...
Fetching P06401 ...
Fetching P06401 ...
Fetching P10275 ...
Fetching P10275 ...
Done: 2B_Nuclear_Ligand_Master_SMILES_do